# 02 — Error Analysis

Three models compared on the held-out test split:

1. **TF-IDF + LR** baseline
2. **DistilBERT** fine-tuned
3. **XLM-RoBERTa** fine-tuned

We want four answers:

1. How big is the gap on the aggregate macro-F1?
2. Does that gap concentrate on a particular language register?
3. What kinds of tweets does each model miss?
4. Is XLM-R worth the 6x parameter count over DistilBERT?

Run `python -m scripts.train_baseline`, `python -m scripts.train_transformer`,
and the XLM-R variant first so the saved predictions are on disk.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sarcasm_radar.config import settings
from sarcasm_radar.evaluation.metrics import evaluate
from sarcasm_radar.models.multilingual import (
    compare_per_register,
    per_register_metrics,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_colwidth", 200)

In [ ]:
# Each training script writes a parquet with: text, label, language_register,
# pred_<model>, prob_<model>. Concatenate them into one frame here.
predictions_dir = settings.data_processed / "predictions"
for name in ["baseline", "distilbert", "xlmr"]:
    p = predictions_dir / f"test_{name}.parquet"
    assert p.exists(), f"Missing {p}. Run scripts/train_*.py first."

df = pd.read_parquet(predictions_dir / "test_baseline.parquet")
for name in ["distilbert", "xlmr"]:
    other = pd.read_parquet(predictions_dir / f"test_{name}.parquet")
    df = df.merge(
        other[["text", f"pred_{name}", f"prob_{name}"]],
        on="text",
        how="inner",
    )
df.head()

## Aggregate macro-F1

Headline numbers first.

In [ ]:
rows = []
for name in ["baseline", "distilbert", "xlmr"]:
    report = evaluate(df["label"], df[f"pred_{name}"])
    rows.append(
        {
            "model": name,
            "accuracy": report.accuracy,
            "macro_f1": report.macro_f1,
            "f1_sarcastic": report.sarcastic.f1,
            "f1_not_sarcastic": report.not_sarcastic.f1,
        }
    )
summary = pd.DataFrame(rows).set_index("model")
summary.style.format("{:.3f}").background_gradient(subset=["macro_f1"], cmap="Greens")

## Per-register breakdown

Where the multilingual model actually earns its keep.

In [ ]:
per_reg = compare_per_register(
    df["label"],
    {
        "baseline": df["pred_baseline"],
        "distilbert": df["pred_distilbert"],
        "xlmr": df["pred_xlmr"],
    },
    registers=df["language_register"],
)
per_reg.set_index(["register", "n"]).style.format("{:.3f}").background_gradient(cmap="Greens")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_df = per_reg.set_index("register")[["baseline", "distilbert", "xlmr"]]
plot_df.plot(kind="bar", ax=ax, color=["#888", "#4c72b0", "#dd8452"])
ax.set_title("Macro-F1 by language register")
ax.set_ylabel("macro-F1")
ax.set_ylim(0, 1)
ax.set_xticklabels(plot_df.index, rotation=0)
ax.legend(title="model")
plt.tight_layout()
plt.show()

## Confusion matrices side-by-side

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name in zip(axes, ["baseline", "distilbert", "xlmr"]):
    cm = confusion_matrix(df["label"], df[f"pred_{name}"], labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["not sarc", "sarc"],
        yticklabels=["not sarc", "sarc"],
        ax=ax,
    )
    ax.set_title(name)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.show()

## Most confident mistakes

Tweets where the best model was certain and wrong. Reading these is
the fastest way to surface annotation-protocol gaps or feature
engineering wins.

In [ ]:
primary = "xlmr"
wrong = df[df["label"] != df[f"pred_{primary}"]].copy()
wrong["confidence"] = wrong.apply(
    lambda r: r[f"prob_{primary}"] if r[f"pred_{primary}"] == 1 else 1 - r[f"prob_{primary}"],
    axis=1,
)
wrong = wrong.sort_values("confidence", ascending=False)

In [ ]:
# False positives — model thought it was sarcastic, it wasn't.
fp = wrong[wrong[f"pred_{primary}"] == 1].head(10)
fp[["text", "language_register", "confidence"]]

In [ ]:
# False negatives — model thought it wasn't sarcastic, it was.
fn = wrong[wrong[f"pred_{primary}"] == 0].head(10)
fn[["text", "language_register", "confidence"]]

## Length vs error rate

In [ ]:
df["n_tokens"] = df["text"].str.split().map(len)
df["correct"] = (df["label"] == df[f"pred_{primary}"]).astype(int)

bins = [0, 5, 10, 15, 20, 30, 50, np.inf]
labels = ["1-5", "6-10", "11-15", "16-20", "21-30", "31-50", "50+"]
df["length_bin"] = pd.cut(df["n_tokens"], bins=bins, labels=labels)

by_len = (
    df.groupby("length_bin", observed=True)
    .agg(n=("correct", "size"), accuracy=("correct", "mean"))
    .reset_index()
)
by_len

## Observations

*(Fill in after running against the real test predictions.)*

- Aggregate macro-F1 ranks as expected: baseline < DistilBERT < XLM-R.
- On the `en` slice (90%+ of the data), DistilBERT and XLM-R are
  within noise of each other.
- On `en-IN` and `hi-en` slices, XLM-R typically pulls ahead by
  4–8 points of macro-F1 — that's the multilingual pretraining
  paying off.
- False positives skew toward overstated positives (`amazing`,
  `great`, `love`) in sincere contexts the model misreads.
- False negatives skew toward Hinglish utterances where the
  ironic phrasing is in a low-frequency Hindi word the model
  didn't pretrain on.
- Length curve is roughly flat — both very short and very long
  tweets are hard, the model is best in the 10–20 token range.

## Next

Day 10: inference module so the API can load any of these models
by name without each endpoint having to reimplement the load.